# Short-Rate Models: Vasicek and CIR

Analytical bond prices and simulation with multiple discretisation schemes.


In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import matplotlib.pyplot as plt
import numpy as np

from rates import cir, vasicek

FIGURES = ROOT / "figures"
FIGURES.mkdir(exist_ok=True)


In [ ]:

a_v, b_v, sigma_v, r0_v = 0.5, 0.04, 0.02, 0.03
a_c, b_c, sigma_c, r0_c = 1.0, 0.04, 0.10, 0.03

maturities = np.linspace(0.5, 10.0, 40)
p_v = [vasicek.bond_price(a_v, b_v, sigma_v, r0_v, 0.0, T) for T in maturities]
p_c = [cir.bond_price(a_c, b_c, sigma_c, r0_c, 0.0, T) for T in maturities]

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(maturities, p_v, label="Vasicek")
ax.plot(maturities, p_c, label="CIR")
ax.set_xlabel("Maturity (years)")
ax.set_ylabel("Zero-coupon bond price P(0,T)")
ax.legend()
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig(FIGURES / "bond_prices.png", dpi=150)
plt.show()


In [ ]:
paths = vasicek.simulate(0.5, 0.04, 0.02, 0.08, 5.0, 100, 20, seed=42)
t = np.linspace(0, 5, paths.shape[1])

fig, ax = plt.subplots(figsize=(8, 4))
for i in range(paths.shape[0]):
    ax.plot(t, paths[i], alpha=0.7)
ax.axhline(0.04, color="k", linestyle="--", label="b")
ax.set_xlabel("Time")
ax.set_ylabel("Short rate r(t)")
ax.set_title("Vasicek sample paths")
ax.legend()
fig.tight_layout()
fig.savefig(FIGURES / "vasicek_paths.png", dpi=150)
plt.show()


In [ ]:
n_steps_list = [5, 10, 20, 40, 80, 160]
errors = {"euler": [], "milstein": []}
reference = cir.simulate(1.0, 0.04, 0.10, 0.03, 1.0, 400, 4000, "exact", seed=0)[:, -1].mean()

for n in n_steps_list:
    for scheme in errors:
        mean = cir.simulate(1.0, 0.04, 0.10, 0.03, 1.0, n, 4000, scheme, seed=0)[:, -1].mean()
        errors[scheme].append(abs(mean - reference))

fig, ax = plt.subplots(figsize=(8, 4))
for scheme, errs in errors.items():
    ax.plot(n_steps_list, errs, marker="o", label=scheme)
ax.set_xlabel("Number of steps")
ax.set_ylabel("|Mean terminal rate - exact reference|")
ax.set_title("CIR discretisation error vs step count")
ax.legend()
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig(FIGURES / "cir_discretization.png", dpi=150)
plt.show()
